# Session 4 — Deutsch, Grover & Shor
### Quantum Computing Workshop · Session 4 of 4 — the finale


- Run cells **top to bottom**.  **YOUR TURN** = fill in a blank. **PREDICT FIRST** = type your prediction *before* running.
- Pairs: driver types, navigator reads aloud. **Swap at Task 3.**

> First: **File → Save a copy in Drive** — this notebook is the last piece of your four-notebook portfolio.

## Cell 1 — Setup (run first, ~1 minute)

In [ ]:
%pip install -q qiskit qiskit-aer matplotlib pylatexenc

from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit.quantum_info import Statevector
import numpy as np
import matplotlib.pyplot as plt
from math import gcd

simulator = AerSimulator()

def run_counts(qc, shots=1000):
    return simulator.run(qc, shots=shots).result().get_counts()

def show_bars(qc, highlight=None):
    """Bar chart of the state's (real) amplitudes."""
    sv = Statevector(qc)
    n = sv.num_qubits
    amps = np.real(sv.data)
    labels = [format(i, f'0{n}b') for i in range(2**n)]
    colors = ['#8B5CF6' if l == highlight else '#00B2CA' for l in labels]
    fig, ax = plt.subplots(figsize=(5.5, 2.6))
    ax.bar(labels, amps, color=colors, width=0.6)
    ax.axhline(np.mean(amps), color='#F5C544', ls='--', lw=2, label='average')
    ax.axhline(0, color='#23244A', lw=0.8)
    ax.set_ylim(-1.05, 1.1); ax.set_ylabel('amplitude'); ax.legend(frameon=False, fontsize=9)
    plt.show()

print("✅ Setup complete — let's cancel out some wrong answers.")

---
## Task 1 — Interrogate the mystery machines 

The cell below builds a **sealed machine known as an oracle**: a gate that is secretly either *quiet* (constant — treats both parts of the state the same) or a *sign-flipper* (balanced — flips the sign of one part). Honor rule: **don't peek at the code's choice.** Your circuit will tell you.

In [ ]:
import random

def mystery_machine():
    """Returns a sealed 1-qubit machine. No peeking at which kind!"""
    machine = QuantumCircuit(1, name="sealed")
    if random.random() < 0.5:
        machine.z(0)          # sign-flipper (balanced)
    else:
        machine.id(0)         # quiet (constant)
    return machine

sealed = mystery_machine()
print("A machine has been sealed. Classically you'd query it twice. You get ONE.")

The test is the **H sandwich** from Gate Golf: `H → machine → H → measure`. **PREDICT FIRST:** which reading means which? (Hint: HZH = X, and H·nothing·H = nothing — Holes 1 and 4 of your scorecard.)

In [ ]:
probe = QuantumCircuit(1, 1)
probe.h(0)
probe.compose(sealed, inplace=True)   # ONE query — the machine runs once
probe.h(0)
probe.measure(0, 0)

verdict = run_counts(probe)
print(verdict)
print("→ VERDICT:", "sign-flipper (balanced)" if '1' in verdict else "constant function")
print("Machine/oracle was in fact:", sealed.data[0].operation.name if sealed.data else "empty")

100% of shots, one query, a certain answer — this is **Deutsch's algorithm (1985)**, the first quantum algorithm ever written, and it's literally the golf identity you already know, used to tell the two machines apart.

Now the 8-case version — the quantum version of your Mystery Deck. A 3-qubit sealed machine is either constant everywhere constant or flips signs on **exactly half** of the 8 basis states (balanced). Your decks charged you 5 flips to be certain; the sandwich charges one:

In [ ]:
def mystery_big():
    machine = QuantumCircuit(3, name="sealed3")
    if random.random() < 0.5:
        machine.z(random.randint(0, 2))   # flips signs on exactly half the 8 states: balanced
    else:
        machine.id(0)                      # constant
    return machine

sealed3 = mystery_big()
probe = QuantumCircuit(3, 3)
probe.h([0, 1, 2])
probe.compose(sealed3, inplace=True)      # one query
probe.h([0, 1, 2])
probe.measure([0, 1, 2], [0, 1, 2])

verdict = run_counts(probe)
print(verdict)
print("→ VERDICT:", "constant" if '000' in verdict else "balanced")

Reading `000` means constant; **anything else** means balanced. One query, any deck size — this is **Deutsch–Jozsa (1992)**.

**YOUR TURN:** run the two cells above a few times each. Keep a tally: does the answer ever come out wrong? (It provably can't — but "provably" is more fun once you've tried to catch it failing.)

<details><summary> What the machine can and can't tell you</summary>

Notice what you *learned*: same-or-different, across all 8 cases. And what you *didn't*: any individual case's answer. The sandwich trades knowing the parts for knowing the pattern. Quantum computers are **tools for finding patterns, not machines that read answers**.
</details>

---
## Task 2 — Read a secret combination in ONE question

The machine below hides a **4-bit secret code**. Classically, learning 4 bits takes 4 questions — one per bit, no shortcuts. Watch the sandwich get all four at once.

In [ ]:
def secret_machine(code):
    """Hides a bit-string as sign flips. (Facilitator builds it; you interrogate it.)"""
    machine = QuantumCircuit(len(code), name="vault")
    for i, bit in enumerate(reversed(code)):
        if bit == '1':
            machine.z(i)
    return machine

vault = secret_machine('1011')   # ← the driver types a secret here; the navigator looks away!

**PREDICT FIRST:** one H-sandwich run — what will the measurement read?

In [ ]:
probe = QuantumCircuit(4, 4)
probe.h(range(4))
probe.compose(vault, inplace=True)        # ONE query
probe.h(range(4))
probe.measure(range(4), range(4))

print(run_counts(probe))

The whole secret, every shot. This is **Bernstein–Vazirani (1992)** — and it's the same physics as Task 1: each secret bit leaves its own sign pattern, and each qubit's sandwich turns that pattern back into its bit. Four separate reads, run as one.

**YOUR TURN:** navigators take the keyboard. Hide a secret your partner hasn't seen (any length — try 6 bits) and let them pull it out in one query. Then answer together: why does this NOT mean "quantum computers read 4 answers at once"?

<details><summary> The honest answer</summary>

The machine's *structure* (one sign per secret bit, applied independently) is exactly the kind of global pattern that one interference pass can read. A machine hiding 4 *unrelated* answers has no such pattern — and no quantum trick reads it in one go. The speedup comes from the structure, not from magic parallelism.
</details>

---
## Task 3 — Grover: watch the wrong answers cancel out
*(Drivers and navigators: swap seats!)*

Four slots, one marked. Classically: ~2.5 flips on average. Grover: **one round, certainty** — and you're going to watch it happen, bar by bar.

**Stage 1 — every answer equal:**

In [ ]:
target = '11'   # ← the item we're hunting (change me later!)

qc = QuantumCircuit(2)
qc.h([0, 1])
show_bars(qc, highlight=target)

**Stage 2 — the mark.** The oracle flips the sign of the target's amplitude — and nothing else. **PREDICT FIRST:** after this, what would a measurement show? (Careful — remember that squaring hides the sign.)

In [ ]:
def mark(qc, target):
    """Flip the sign of |target⟩'s amplitude. cz flips |11⟩; the x's aim it at the chosen target."""
    flips = [i for i, b in enumerate(reversed(target)) if b == '0']
    for i in flips: qc.x(i)
    qc.cz(0, 1)
    for i in flips: qc.x(i)

mark(qc, target)
show_bars(qc, highlight=target)

Marked — and yet **measurement would still show 25/25/25/25**, because squaring erases the sign. The mark is invisible... until:

**Stage 3 — reflect about the average.** Every bar flips to its mirror image across the gold average line. Do the arithmetic *before* running: the bars are (0.5, 0.5, 0.5, −0.5), average 0.25. Where does each one land?

In [ ]:
def reflect_about_average(qc):
    """The diffuser: an H/X sandwich around a sign flip — every amplitude reflects across the mean."""
    qc.h([0, 1])
    qc.x([0, 1])
    qc.cz(0, 1)
    qc.x([0, 1])
    qc.h([0, 1])

reflect_about_average(qc)
show_bars(qc, highlight=target)

Wrong answers at **exactly zero** — cancelled out completely — and the target at 100% (a bar of ±1: the overall sign can't be measured, only the size counts). Confirm with real shots:

In [ ]:
qcm = qc.copy()
qcm.measure_all()
print(run_counts(qcm))

**YOUR TURN:** change `target` at the top of this task to `'00'`, `'01'`, or `'10'` and re-run the four cells. Every needle, one round, certainty.

**The number of rounds.** Bigger haystacks need more rounds — but only about (π/4)·√N of them, and *no more*. Watch what happens on 8 slots as you keep going past the right number:

In [ ]:
def grover8(rounds, target='101'):
    qc = QuantumCircuit(3)
    qc.h(range(3))
    for _ in range(rounds):
        flips = [i for i, b in enumerate(reversed(target)) if b == '0']
        for i in flips: qc.x(i)
        qc.h(2); qc.ccx(0, 1, 2); qc.h(2)      # 3-qubit sign flip
        for i in flips: qc.x(i)
        qc.h(range(3)); qc.x(range(3))
        qc.h(2); qc.ccx(0, 1, 2); qc.h(2)
        qc.x(range(3)); qc.h(range(3))
    return float(np.abs(Statevector(qc).data[int(target, 2)])**2)

rounds = range(1, 7)
probs = [grover8(r) for r in rounds]
for r, p in zip(rounds, probs):
    print(f"{r} round(s): P(find it) = {p:.1%}")
plt.plot(list(rounds), probs, 'o-', color='#8B5CF6'); plt.axhline(1/8, ls='--', color='#5A6478')
plt.xlabel('Grover rounds'); plt.ylabel('P(target)'); plt.title('Probability by Round'); plt.show()

<details><summary>What the chart is telling you</summary>

~78% after 1 round, **94.5% after 2** (the sweet spot for 8 slots), then *down* — 33%, then ~1%! Grover is a rotation with a right number of rounds, not a ratchet: keep reflecting past the peak and the answer's amplitude swings back down. "More quantum" is not "more better" — you have to stop at the right round. (The dashed line is 1/8 — blind guessing. And that recovery at round 6? The process is periodic — miss the peak and you *can* catch the next one, at three times the cost. A lot of quantum algorithms fail simply because someone miscounted.)
</details>

---
## Task 4 — Factor like Shor (the by-hand part, automated) 
We factored 15 and 21 using only a **period** and two **gcd**s. Here's that exact procedure as code — with the one quantum-sized step labeled.

In [ ]:
def find_period(a, N):
    """THE QUANTUM STEP. For big N this is the slow part —
    the ONLY part Shor's algorithm hands to a quantum computer,
    which finds it by interference (a scaled-up H-sandwich / quantum Fourier transform).
    For small N, brute force works fine:"""
    x = 1
    for r in range(1, N):
        x = (x * a) % N
        if x == 1:
            return r
    return None

def factor_by_period(N):
    for a in [2, 3, 5, 7, 11, 13]:
        if gcd(a, N) != 1:
            print(f"  base {a}: shares a factor with {N} outright — lucky day: {gcd(a, N)}")
            return gcd(a, N), N // gcd(a, N)
        r = find_period(a, N)
        print(f"  base {a}: period of {a}^x mod {N} is {r}", end="")
        if r % 2 == 1:
            print("  → odd period, retry with a new base"); continue
        h = pow(a, r // 2, N)
        if h == N - 1:
            print("  → unlucky (half-power ≡ −1), retry with a new base"); continue
        p, q = gcd(h - 1, N), gcd(h + 1, N)
        print(f"  → gcd({h}−1, {N}) = {p},  gcd({h}+1, {N}) = {q}")
        return p, q
    return None

for N in [15, 21]:
    print(f"Factoring {N}:")
    p, q = factor_by_period(N)
    print(f"  ✔ {N} = {p} × {q}\n")

**YOUR TURN:** factor **33**, then **91**, then the biggest number you dare (try 143, 187, 247…). Watch the retry logic earn its keep — some bases are duds, exactly as the worksheet warned.

<details><summary> What to notice</summary>

For 33, base 2 hits the "half-power ≡ −1" dud and the code moves on to a base that works — real Shor does the same, and a couple of retries almost always suffice. What the printout shows you is the honest anatomy of Shor's algorithm: **everything is easy classical arithmetic except `find_period`** — and for a 600-digit N, that period is astronomically long, which is exactly where the quantum computer comes in. You now understand, structurally, the algorithm that started the encryption clock.
</details>

---
---
# Exploration (optional)

### Task E1 — Build a vault, challenge a neighbor
Design your own sealed machine (any of today's kinds — constant/balanced, or a secret code) and pass your notebook to the next pair. They get ONE query. Referee each other's honor rules.

In [ ]:
# your vault here


### Task E2 — TWO needles in 8 slots
Modify the Task 3 rhythm experiment: mark **two** targets (call `mark`-style sign flips for both inside the loop). **PREDICT FIRST:** more needles — does the best round count go up or down?

In [ ]:
# your two-needle Grover here


<details><summary>Solution sketch & the surprise</summary>

Flip signs for both targets each round (two aimed sign-flips), same reflect step. The surprise: with 2 marked of 8, **one single round** lands a marked item with 100% certainty (split between the two needles). More needles = a bigger head start = *fewer* rounds — the beat count is ~(π/4)·√(N/k) for k needles. Grover rewards easier problems, quantitatively.
</details>

### Task E3 — The period race
Time `factor_by_period` on growing N (`%time factor_by_period(8633)` — that's 89 × 97). Roughly how does the classical period hunt grow as N gets bigger? Now imagine N with 600 digits — write one sentence to your past self from Session 1 explaining the 300-trillion-years number.

In [ ]:
# your race here


### Task E4 — When your laptop taps out
A quantum computer with n qubits juggles 2ⁿ amplitudes. Your simulator stores them all. Find your machine's limit: build `QuantumCircuit(n)`, apply `h` to every qubit, and time `Statevector(qc)` for n = 10, 15, 20, 24… (Stop before ~28 unless you enjoy restarting runtimes.) Each +1 qubit **doubles** the memory — write down where Colab gives up, and appreciate why real qubits are worth all that refrigeration.

In [ ]:
# your scaling experiment here


---
## "Graduation"


**Where to next (all free):**
- **IBM Quantum Learning** (learning.quantum.ibm.com) — structured courses picking up exactly here: error correction, real algorithm design, badges employers recognize.
- **The open Qiskit courses & YouTube channel** — the community's teaching home.
- **Your own machine time** — the real-hardware free tier you used in Session 3 is yours to keep using.

You know what a qubit is, physically. You can read a state/recipe, drive the bloch sphere, verify an identity, quantify hardware noise, and explain — correctly — what quantum computers actually do: *they arrange for the wrong answers to cancel out.* Great job! 